AuthorityEffect: None.


## 4. Student Exercises

To solidify your understanding, complete the following three exercises:

### Exercise 1: 4-Cycle Inspection
Given a 4-cycle of items $W \to X \to Y \to Z \to W$ with claimed edge differences:
- $W \to X = +2$
- $X \to Y = +3$
- $Y \to Z = -1$
- $Z \to W = -4$

Compute the cycle sum and defect $\Phi_1$. Is this system `CONSISTENT` or a `CONTRADICTION`?

### Exercise 2: Detecting Field Injection in a Sealed Packet
An untrusted model returns a payload that introduces an unauthorized field `"self_certified": true`.
Write a function `validate_packet_schema` that rejects any payload containing keys outside an approved schema allowlist `{"step", "vertex_from", "vertex_to", "value"}`.

### Exercise 3: Potential Gauge Reconstruction
If a cycle of 3 vertices is consistent with differences:
- $A \to B = 5$
- $B \to C = -2$
- $C \to A = -3$

Assuming gauge anchor $x_A = 0$, compute the exact scalar potentials $x_B$ and $x_C$.


In [ ]:
# [Pinned] Environment verification & basic dependencies
import sys
from dataclasses import dataclass
from fractions import Fraction
from pathlib import Path
from typing import List, Tuple

assert sys.version_info >= (3, 10), "Requires Python >= 3.10"
print(f"[OK] Python version: {sys.version.split()[0]}")

In [ ]:
import sys

_REPO = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "engine/crates_rs/core_functional").is_dir()
)
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))
from adversary.infrastructure.providers.adapters.native_functional import compute_graph_phi1

# [Proved] The 3-Cycle Invariant: Summing differences around a closed loop
claimed_differences = {
    ("A", "B"): Fraction(1),
    ("B", "C"): Fraction(1),
    ("C", "A"): Fraction(1),
}

cycle = [("A", "B"), ("B", "C"), ("C", "A")]
cycle_sum = sum(claimed_differences[edge] for edge in cycle)

print(f"Cycle: {' -> '.join(e[0] for e in cycle)} -> {cycle[0][0]}")
print(f"Sum around cycle: {cycle_sum}")

# The host computes the cycle defect Phi_1
phi_1, _gauge = compute_graph_phi1(
    ["A", "B", "C"], [(u, v, 1) for u, v in cycle], claimed_differences
)
verdict = "CONTRADICTION" if phi_1 > 0 else "CONSISTENT"

print(f"Defect Phi_1: {phi_1}")
print(f"Host Verdict: {verdict}")

assert phi_1 == Fraction(3), f"Expected defect 3, got {phi_1}"
assert verdict == "CONTRADICTION", "Cycle must be recognized as contradictory"
print("[OK] 3-cycle defect computed and verified!")

In [ ]:
import sys
from pathlib import Path

_REPO = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "engine/crates_rs/core_functional").is_dir()
)
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))
from adversary.infrastructure.providers.adapters.native_functional import compute_graph_phi1

# [Declared] Implementation of a Minimal Layer-3 Constraint Host Referee


@dataclass(frozen=True)
class EdgeConstraint:
    source: str
    target: str
    difference: Fraction  # target - source


class MinimalHostReferee:
    """A minimal Layer-3 Host Referee for directed cycle consistency."""

    @staticmethod
    def evaluate_cycle(constraints: List[EdgeConstraint]) -> Tuple[Fraction, str]:
        """Calculates exact rational defect Phi_1 and returns a fail-closed verdict."""
        n = len(constraints)
        for i in range(n):
            curr_target = constraints[i].target
            next_source = constraints[(i + 1) % n].source
            if curr_target != next_source:
                raise ValueError(f"Disconnected cycle at {curr_target} != {next_source}")

        sum(c.difference for c in constraints)
        phi_1, _gauge = compute_graph_phi1(
            list(dict.fromkeys(c.source for c in constraints)),
            [(c.source, c.target, 1) for c in constraints],
            {(c.source, c.target): c.difference for c in constraints},
        )
        verdict = "CONSISTENT" if phi_1 == 0 else "CONTRADICTION"
        return phi_1, verdict


referee = MinimalHostReferee()

# Case A: The contradictory triangle from Section 1
triangle_bad = [
    EdgeConstraint("A", "B", Fraction(1)),
    EdgeConstraint("B", "C", Fraction(1)),
    EdgeConstraint("C", "A", Fraction(1)),
]
phi_bad, verdict_bad = referee.evaluate_cycle(triangle_bad)
print(f"Case A (Inconsistent): Defect={phi_bad}, Verdict={verdict_bad}")
assert verdict_bad == "CONTRADICTION"

# Case B: A consistent triangle (e.g. x_A=0, x_B=1, x_C=3 => diffs: +1, +2, -3)
triangle_good = [
    EdgeConstraint("A", "B", Fraction(1)),
    EdgeConstraint("B", "C", Fraction(2)),
    EdgeConstraint("C", "A", Fraction(-3)),
]
phi_good, verdict_good = referee.evaluate_cycle(triangle_good)
print(f"Case B (Consistent):   Defect={phi_good}, Verdict={verdict_good}")
assert verdict_good == "CONSISTENT"
print("[OK] Layer-3 host referee verified both consistent and inconsistent cases!")

In [ ]:
# [Proved] Student Exercise Solutions & Automated Verification

# Solution 1: 4-Cycle Inspection
cycle_4_constraints = [
    EdgeConstraint("W", "X", Fraction(2)),
    EdgeConstraint("X", "Y", Fraction(3)),
    EdgeConstraint("Y", "Z", Fraction(-1)),
    EdgeConstraint("Z", "W", Fraction(-4)),
]
phi_4, verd_4 = referee.evaluate_cycle(cycle_4_constraints)
print(f"[Exercise 1] 4-Cycle: Defect={phi_4}, Verdict={verd_4}")
assert phi_4 == Fraction(0)
assert verd_4 == "CONSISTENT"
print("[OK] Exercise 1 verified!")

# Solution 2: Schema Field Allowlist
ALLOWED_KEYS = {"step", "vertex_from", "vertex_to", "value"}


def validate_packet_schema(payload: dict) -> bool:
    """Refuses any payload containing unauthorized or injected fields."""
    return set(payload.keys()).issubset(ALLOWED_KEYS)


valid_payload = {"step": 1, "vertex_from": "A", "vertex_to": "B", "value": 10}
injected_payload = {
    "step": 1,
    "vertex_from": "A",
    "vertex_to": "B",
    "value": 10,
    "self_certified": True,
}

assert validate_packet_schema(valid_payload) is True
assert validate_packet_schema(injected_payload) is False
print("[OK] Exercise 2 verified!")

# Solution 3: Gauge Reconstruction
x_A = Fraction(0)
x_B = x_A + Fraction(5)
x_C = x_B + Fraction(-2)

assert x_A - x_C == Fraction(-3)
print(f"[Exercise 3] Reconstructed Potentials: x_A={x_A}, x_B={x_B}, x_C={x_C}")
print("[OK] Exercise 3 verified!")

## 5. Summary & Next Steps

### Key Beginner Takeaways
1. **The Naked Prompt Trap**: Language models predict likely tokens; they do not natively verify discrete global invariants. Persuasive prose can and will launder mathematical contradictions.
2. **Three-Layer Sovereignty**: Always isolate continuous generation (Layers 1 & 2) from discrete external verification (Layer 3).
3. **The Cycle Invariant ($\Phi_1$)**: Closed loops in directed difference networks must sum to zero. Any non-zero sum is an incontrovertible witness of defect ($\Phi_1 > 0$).
4. **Sealed Packets**: Never trust raw strings across system seams. Cryptographic envelope sealing prevents data tampering and prompt injection.

### Where to Go Next
- **Track 1 (Practitioners)**: Dive into `lesson_970_07_section_2_daily_workflow_integrations` and `lesson_970_08_epistemic_failure_classes` to learn how to build real Claude Code pre-tool hooks, Codex linters, Antigravity multi-agent swarms, and the 12 Epistemic Failure Classes.
- **Track 2 (Foundations)**: Explore `700_math/` for the full mathematical theory of simplicial cochain complexes, sheaf cohomology, discrete $L_1$ coboundary projection, and totally unimodular matrices.


> **Claim-typing legend**
> - **[Pinned]**: taken from a file this repository already has.
> - **[Proved]**: a finite check in this notebook. Not a general proof.
> - **[Declared]**: a carrier, rule, or heading chosen for this sitting.
> - **[Open]**: something this sitting does not decide.
## Anchors

- Former notebook: `00_symbol_game_for_beginners`
- The code checks one finite example. It is not a general proof.
